# 04 — Evaluation
Load the trained baseline, score it on the sealed test set, judge it honestly.
Input:  models/baseline/model.joblib  +  raw_data/processed/test.csv
Output: models/baseline/metrics.json


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd, json
import joblib, numpy as np
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
from bparadigm.paths import BASELINE_MODEL, TEST_CSV, BASELINE_METRICS

In [3]:
pipe   = joblib.load(BASELINE_MODEL)
test   = pd.read_csv(TEST_CSV)
print("test:", test.shape)
test["label"].value_counts(normalize=True).round(3)   # ~0.72/0.28 — the fair mirror stratify gave you

test: (3917, 2)


label
0    0.723
1    0.277
Name: proportion, dtype: float64

In [4]:
y_true = test["label"]
y_pred = pipe.predict(test["text"])

In [5]:
majority = y_true.value_counts().idxmax()                 # the dumb guess: always predict Negative
floor_acc = accuracy_score(y_true, [majority] * len(y_true))
print(f"majority-class floor: {floor_acc:.3f} accuracy (The model always predicts {majority})")

majority-class floor: 0.723 accuracy (The model always predicts 0)


In [6]:
acc    = accuracy_score(y_true, y_pred)
pos_f1 = f1_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
macro  = f1_score(y_true, y_pred, average="macro", zero_division=0)
print(f"accuracy    : {acc:.3f}")
print(f"positive-F1 : {pos_f1:.3f}")
print(f"macro-F1    : {macro:.3f}")

accuracy    : 0.952
positive-F1 : 0.913
macro-F1    : 0.940


In [7]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
print("rows = true, cols = pred   [Negative, Positive]")
print(cm)

rows = true, cols = pred   [Negative, Positive]
[[2743   89]
 [  98  987]]


In [8]:
print(classification_report(y_true, y_pred,
      target_names=["Negative", "Positive"], digits=3, zero_division=0))

              precision    recall  f1-score   support

    Negative      0.966     0.969     0.967      2832
    Positive      0.917     0.910     0.913      1085

    accuracy                          0.952      3917
   macro avg      0.941     0.939     0.940      3917
weighted avg      0.952     0.952     0.952      3917



In [9]:
vec, clf = pipe.named_steps["tfidf"], pipe.named_steps["clf"]
feats = np.array(vec.get_feature_names_out()); coef = clf.coef_[0]
print("most POSITIVE:", ", ".join(feats[np.argsort(coef)[-12:][::-1]]))
print("most NEGATIVE:", ", ".join(feats[np.argsort(coef)[:12]]))

most POSITIVE: great, always, love, good, best, excellent, fast, amazing, easy, the best, thank, thank you
most NEGATIVE: not, worst, to, now, they, no, account, terrible, don, poor, horrible, then


In [10]:
metrics = {"accuracy": round(acc, 4), "positive_f1": round(pos_f1, 4),
           "macro_f1": round(macro, 4), "floor_accuracy": round(floor_acc, 4),
           "confusion_matrix": cm.tolist(), "labels": ["Negative", "Positive"]}
(BASELINE_METRICS).write_text(json.dumps(metrics, indent=2))
print("saved metrics.json:", metrics)

saved metrics.json: {'accuracy': 0.9523, 'positive_f1': 0.9135, 'macro_f1': 0.9403, 'floor_accuracy': 0.723, 'confusion_matrix': [[2743, 89], [98, 987]], 'labels': ['Negative', 'Positive']}
